# Jev off-the-shelf calibration: known probabilities + Monte Carlo

First benchmark: **raw Jev outputs, no fine-tuning and no post-hoc calibration**. Compare Choice and Noul on a biased-coin sweep and threshold events from eight other distributions. Source: [Berryman coin diagnostic](https://www.linkedin.com/feed/update/urn:li:activity:7506796921362030593/), [API contract](https://docs.typesafe.ai/api), [versioned models](https://docs.typesafe.ai/models), and our [Monte Carlo protocol](../monte-carlo-calibration.md).

The API sees synthetic scenarios only. Credentials are read inside the client from the project's owner-only `.env`; never put keys in cells, display environment variables, or enable verbose HTTP/debug tracebacks. Cached files contain synthetic prompts, validated probabilities, model ID, timing and token counts—not raw HTTP responses or headers. API calls are disabled by default and completed scenarios are reused.


In [ ]:
%xmode Plain
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from jev_research.jev_api import BenchmarkError, collect, configuration
from jev_research.monte_carlo import evaluate, scenarios

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/uv.sh").exists())
RUN_LIVE = False  # Set True only when you want to send the missing scenarios to Jev.
MAX_REQUESTS = 80
TRIALS_PER_SCENARIO = 2_000
MC_REPETITIONS = 500
SEED = 42
settings = configuration(ROOT)  # Contains model ID and key-presence boolean only.
print(
    {
        "model": settings["model"],
        "key_present": settings["key_present"],
        "live_calls_enabled": RUN_LIVE,
    }
)

## 1. Freeze the scenario grid

63 requests maximum for the initial grid, three questions per request: Choice, Noul(event), and Noul(complement). Each distribution-parameter task has a paired version stating the exact event probability, separating probability preservation from arithmetic. The coin grid includes endpoints and near-ties. Every prediction concerns a future random outcome, not which outcome is most likely.

This first pass uses packed questions and one wording per scenario. It is a diagnostic, not an independent replication of the original post. Separate-request and paraphrase ablations follow if needed. The hand-designed grid is not a deployment distribution.


In [ ]:
cases = scenarios()
manifest = pd.DataFrame(cases)
display(
    manifest.groupby(["family", "track"]).agg(
        scenarios=("q", "size"), min_q=("q", "min"), max_q=("q", "max")
    )
)
print(f"At most {len(cases)} HTTP requests / {3 * len(cases)} questions; no automatic retries.")

## 2. Collect or resume cached predictions

Put `TYPESAFE_API_KEY` in the project `.env` and set `RUN_LIVE=True` above to run. The pinned model is `jev-1.13.0`; the client rejects a different response model or malformed probabilities. A failed call stops collection with a sanitized error and preserves completed scores. A timeout may have been charged: retries are manual. No live calls are needed for Monte Carlo resampling.


In [ ]:
try:
    rows = collect(ROOT, cases, live=RUN_LIVE, max_requests=MAX_REQUESTS)
except BenchmarkError as exc:
    print(str(exc))  # Only this adapter's sanitized errors are displayed.
    rows = collect(ROOT, cases, live=False)
complete = len(rows) == len(cases)
print({"cached_scenarios": len(rows), "planned_scenarios": len(cases), "complete": complete})
if rows:
    predictions = pd.DataFrame(rows)
    display(predictions[["family", "track", "q", "choice_p", "noul_p", "complement_p"]])
    print(
        {
            "cached_input_tokens": int(predictions.input_tokens.sum()),
            "note": "Total for cached calls, not current-run billing.",
        }
    )
else:
    print(
        "No Jev results yet. Add the key to .env and enable RUN_LIVE; "
        "no simulated model scores are substituted."
    )

## 3. Probability recovery and cross-interface coherence

Read `Choice.probabilities`, not the API's derived `confidence` statistic. Compare both primitives to the oracle event probability. For negated Noul, inspect `P(event)+P(complement)-1`. These are probability-recovery and coherence diagnostics; neither alone is a calibration measure.


In [ ]:
if rows:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, track in zip(axes[:2], ["explicit", "parameters"], strict=True):
        panel = predictions[predictions.track == track]
        ax.scatter(panel.q, panel.choice_p, label="Choice", alpha=0.7)
        ax.scatter(panel.q, panel.noul_p, label="Noul", marker="x")
        ax.plot([0, 1], [0, 1], "k--", label="Oracle")
        ax.set(
            xlabel="True event probability",
            ylabel="Jev probability",
            title=track,
            xlim=(-0.02, 1.02),
            ylim=(-0.02, 1.02),
        )
        ax.legend()
    coin = predictions[predictions.family == "bernoulli"].sort_values("q")
    axes[2].plot(coin.q, coin.choice_p, "o-", label="Choice")
    axes[2].plot(coin.q, coin.noul_p, "x-", label="Noul")
    axes[2].plot([0, 1], [0, 1], "k--")
    axes[2].set(title="Biased coin sweep", xlabel="P(heads)", ylabel="Jev P(heads)")
    axes[2].legend()
    plt.suptitle(f"Raw {settings['model']} — {'complete' if complete else 'PARTIAL'} grid")
    plt.tight_layout()
    plt.show()
    display(
        pd.DataFrame(
            {
                "family": predictions.family,
                "track": predictions.track,
                "choice_noul_gap": abs(predictions.choice_p - predictions.noul_p),
                "complement_gap": abs(predictions.noul_p + predictions.complement_p - 1),
            }
        )
        .groupby(["family", "track"])
        .mean()
    )

## 4. Monte Carlo outcomes and proper scores

For each fixed scenario with true probability `q`, sample 2,000 Bernoulli outcomes, repeated 500 times with a fixed seed. Binomial counts are equivalent to simulating each trial and use less memory. Choice and Noul share identical outcome draws for paired comparisons.

We compare empirical Brier/log loss with their analytic expectations. Exact zero/one model predictions can produce **infinite** log loss; the detail table separately includes an explicitly clipped sensitivity score. The Monte Carlo ranges below measure outcome sampling noise conditional on these prompts and fixed model scores—not uncertainty across real-world tasks, model retraining, or repeated API calls. Outcomes for each scenario are generated separately; this initial notebook does not estimate a joint distribution over related threshold questions.


In [ ]:
details, summary, reliability = evaluate(
    rows, trials=TRIALS_PER_SCENARIO, repetitions=MC_REPETITIONS, seed=SEED
)
if rows:
    display(summary)
    display(
        details.groupby(["family", "track", "primitive"])[
            [
                "absolute_probability_error",
                "expected_brier",
                "expected_nll",
                "oracle_expected_brier",
                "oracle_expected_nll",
            ]
        ].mean()
    )
    print(
        "MC p025/p975 columns describe repeated outcome draws; "
        "they are not deployment confidence intervals."
    )

## 5. Reliability diagrams

Group predictions into 10 equal-width probability bins, separately by primitive and prompt track. Plot the actual mean predicted probability against the mean oracle event probability; show Monte Carlo event frequencies and their central 95% ranges. Empty bins are omitted and counts refer to scenarios. A model can be calibrated after coarsening while failing to recover individual known probabilities, so inspect both this chart and Section 3.


In [ ]:
if rows:
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, (primitive, track) in zip(
        axes.flat,
        [(p, t) for p in ["choice", "noul"] for t in ["explicit", "parameters"]],
        strict=True,
    ):
        panel = reliability[(reliability.primitive == primitive) & (reliability.track == track)]
        ax.plot([0, 1], [0, 1], "k--")
        ax.plot(panel.mean_prediction, panel.oracle_event_rate, "o-", label="Oracle bin rate")
        ax.vlines(
            panel.mean_prediction,
            panel.mc_p025,
            panel.mc_p975,
            color="orange",
            label="MC central 95%",
        )
        ax.scatter(panel.mean_prediction, panel.mc_event_rate, color="orange", marker="x")
        for row in panel.itertuples():
            ax.annotate(
                f"n={row.scenarios}", (row.mean_prediction, row.oracle_event_rate), fontsize=8
            )
        ax.set(
            title=f"{primitive} / {track}",
            xlabel="Mean predicted probability",
            ylabel="Event frequency",
            xlim=(-0.03, 1.03),
            ylim=(-0.03, 1.03),
        )
        ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Export and interpret

Outputs stay under Git-ignored `runs/jev-off-the-shelf/`. Report failures/incomplete coverage, model version, prompt track, and raw versus derived metrics. Keep this initial benchmark raw; use independent calibration/test scenarios in a later notebook if fitting corrections. No claim about general-purpose calibration follows from a small synthetic grid.


In [ ]:
if rows:
    out = ROOT / "runs" / "jev-off-the-shelf" / settings["model"] / "analysis"
    out.mkdir(parents=True, exist_ok=True)
    predictions.to_csv(out / "predictions.csv", index=False)
    details.to_csv(out / "scenario_metrics.csv", index=False)
    summary.to_csv(out / "summary.csv", index=False)
    reliability.to_csv(out / "reliability.csv", index=False)
    run_manifest = {
        "model": settings["model"],
        "seed": SEED,
        "trials_per_scenario": TRIALS_PER_SCENARIO,
        "mc_repetitions": MC_REPETITIONS,
        "completed_scenarios": len(rows),
        "complete": complete,
        "planned_scenarios": len(cases),
        "posthoc_calibration": False,
        "cases": cases,
    }
    (out / "manifest.json").write_text(json.dumps(run_manifest, indent=2) + "\n")
    print("Saved local numeric results and synthetic scenario manifest.")